# Interactive ChlorA Heatmap — CalCOFI
Trains a model (RF vs Gradient Boosting) to predict Chlorophyll-A from location-independent oceanographic features, then renders an interactive heatmap over the California coast with sliders for the most important variables.

In [1]:
# Install dependencies if needed
%pip install scikit-learn plotly ipywidgets pandas numpy anywidget

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score

import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_csv('../Data/Finalized_CalCOFI_Phytoplankton.csv')

# Candidate features — now includes Lat_Dec/Lon_Dec so the model learns
# location-based patterns. Spatial features are used as per-cell inputs
# in the heatmap (not exposed as sliders).
CANDIDATE_FEATURES = [
    'T_degC', 'Salnty', 'O2ml_L', 'O2Sat',
    'PO4uM', 'SiO3uM', 'NO3uM', 'Cloud_Amt', 'NP_Ratio', 'Phaeop',
    'Lat_Dec', 'Lon_Dec'
]
TARGET = 'ChlorA'

# Spatial features — injected per grid cell in heatmap, never sliders
SPATIAL_FEATURES = ['Lat_Dec', 'Lon_Dec']

df_clean = df[CANDIDATE_FEATURES + [TARGET]].dropna()
X_all = df_clean[CANDIDATE_FEATURES]
y     = df_clean[TARGET]

print(f"Samples after dropping NaN: {len(df_clean):,}")
print(f"ChlorA range: {y.min():.3f} – {y.max():.3f} μg/L  |  mean: {y.mean():.3f}")

Samples after dropping NaN: 32,452
ChlorA range: 0.000 – 31.280 μg/L  |  mean: 0.906


In [3]:
# ── Compare Random Forest vs Gradient Boosting (5-fold CV) ───────────────────
print("Evaluating Random Forest (5-fold CV)...")
rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
rf_rmse = -cross_val_score(rf, X_all, y, cv=5,
                            scoring='neg_root_mean_squared_error',
                            n_jobs=-1).mean()

print("Evaluating Gradient Boosting (5-fold CV)...")
gb = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                max_depth=5, random_state=42)
gb_rmse = -cross_val_score(gb, X_all, y, cv=5,
                            scoring='neg_root_mean_squared_error').mean()

print(f"\nRandom Forest  CV-RMSE: {rf_rmse:.4f}")
print(f"Gradient Boost CV-RMSE: {gb_rmse:.4f}")

# Pick the model with lower RMSE and fit on full data
if rf_rmse <= gb_rmse:
    best_model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
    model_name = "Random Forest"
else:
    best_model = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                            max_depth=5, random_state=42)
    model_name = "Gradient Boosting"

print(f"\n→ Selected: {model_name}")
best_model.fit(X_all, y)

Evaluating Random Forest (5-fold CV)...
Evaluating Gradient Boosting (5-fold CV)...

Random Forest  CV-RMSE: 1.0664
Gradient Boost CV-RMSE: 1.0652

→ Selected: Gradient Boosting


,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'squared_error'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.05
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",300
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are""friedman_mse"" for the mean squared error with improvement score byFriedman, ""squared_error"" for mean squared error. The default value of""friedman_mse"" is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",5
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft

In [4]:
# ── Feature importance — select >5% OR top 6, whichever gives more features ──
importances = pd.Series(
    best_model.feature_importances_, index=CANDIDATE_FEATURES
).sort_values(ascending=False)

print("Feature Importances:")
for feat, imp in importances.items():
    bar = '█' * int(imp * 60)
    flag = ' ◀ selected' if imp >= 0.05 else ''
    print(f"  {feat:12s}  {imp:.3f}  {bar}{flag}")

over_threshold = importances[importances >= 0.05].index.tolist()
sig_features = over_threshold if len(over_threshold) >= 6 else importances.head(6).index.tolist()

print(f"\nFinal selection ({len(sig_features)} features): {sig_features}")

# Retrain on selected features only
best_model.fit(X_all[sig_features], y)
final_rmse = -cross_val_score(best_model, X_all[sig_features], y, cv=5,
                               scoring='neg_root_mean_squared_error').mean()
print(f"Final model CV-RMSE (selected features): {final_rmse:.4f}")

Feature Importances:
  Phaeop        0.721  ███████████████████████████████████████████ ◀ selected
  O2ml_L        0.065  ███ ◀ selected
  Salnty        0.055  ███ ◀ selected
  T_degC        0.044  ██
  PO4uM         0.035  ██
  NP_Ratio      0.026  █
  O2Sat         0.019  █
  SiO3uM        0.018  █
  NO3uM         0.012  
  Cloud_Amt     0.006  

Final selection (6 features): ['Phaeop', 'O2ml_L', 'Salnty', 'T_degC', 'PO4uM', 'NP_Ratio']
Final model CV-RMSE (selected features): 1.0552


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from global_land_mask import globe 

# ── 0. DATA SAFETY CHECK ───────────────────────────────────────────────────
# Ensure we have the necessary variables from your training cells
if 'sig_features' not in locals():
    try:
        sig_features = X_all.columns.tolist()
    except NameError:
        raise NameError("X_all is not defined. Please run your data loading cells first.")

# ── 1. Build prediction grid and MASK LAND ────────────────────────────────────
LAT_MIN, LAT_MAX = 29.5, 35.5
LON_MIN, LON_MAX = -122.5, -116.5
RESOLUTION = 0.1

lat_pts = np.arange(LAT_MIN, LAT_MAX + RESOLUTION, RESOLUTION)
lon_pts = np.arange(LON_MIN, LON_MAX + RESOLUTION, RESOLUTION)
lon_grid, lat_grid = np.meshgrid(lon_pts, lat_pts)

# Create land mask
is_on_land = globe.is_land(lat_grid, lon_grid)

# Keep only water points
grid_lats = lat_grid[~is_on_land].ravel()
grid_lons = lon_grid[~is_on_land].ravel()

# ── 2. Prediction Logic ───────────────────────────────────────────────────────
# Slider features = all significant features EXCEPT spatial ones
slider_features = [f for f in sig_features if f not in SPATIAL_FEATURES]
feature_means = X_all[slider_features].mean()

def predict_grid(**slider_vals):
    n = len(grid_lats)
    # Fill non-spatial features from slider values (or their means as default)
    data = {feat: np.full(n, slider_vals.get(feat, feature_means[feat]))
            for feat in slider_features}
    # Inject per-cell lat/lon — each grid point gets its own coordinates
    data['Lat_Dec'] = grid_lats
    data['Lon_Dec'] = grid_lons
    # Reorder columns to match training feature order
    preds = best_model.predict(pd.DataFrame(data)[sig_features])
    return np.clip(preds, 0, None)

init_vals  = {feat: float(feature_means[feat]) for feat in slider_features}
init_preds = predict_grid(**init_vals)

# Set color scale limits based on your actual target variable y
try:
    chlora_p95 = float(np.percentile(y, 95))
except NameError:
    chlora_p95 = float(np.max(init_preds))

# ── 3. Plotly FigureWidget ────────────────────────────────────────────────────
# Note: If this still errors, restart your kernel after pip installing anywidget
fig = go.FigureWidget(data=[go.Densitymapbox(
    lat=grid_lats,
    lon=grid_lons,
    z=init_preds,
    radius=12,         
    colorscale=[[0, 'rgba(0,0,0,0)'], [1, 'rgba(0,0,0,1)']],
    reversescale=False,
    colorbar=dict(title='ChlorA<br>(μg/L)', thickness=16),
    zmin=0,
    zmax=chlora_p95,
    hovertemplate='Lat: %{lat:.2f}<br>Lon: %{lon:.2f}<br>Predicted ChlorA: %{z:.3f} μg/L<extra></extra>',
)])

fig.update_layout(
    mapbox_style="white-bg", 
    mapbox_layers=[{
        "below": 'traces',
        "sourcetype": "raster",
        "source": ["https://basemaps.cartocdn.com/light_all/{z}/{x}/{y}.png"]
    }],
    mapbox_center={"lat": 32.5, "lon": -119.5},
    mapbox_zoom=5.5,
    title=dict(text="Ocean-Only Predicted Chlorophyll-A",
               x=0.5, font=dict(size=14)),
    height=620,
    margin=dict(l=0, r=0, t=45, b=0),
)

# ── 4. Sliders (spatial features excluded — they are per-cell inputs) ──────────
sliders = {}
for feat in slider_features:
    col = X_all[feat]
    q05, q95 = col.quantile(0.05), col.quantile(0.95)
    step = (q95 - q05) / 100
    sliders[feat] = widgets.FloatSlider(
        value=round(float(feature_means[feat]), 4),
        min=round(float(q05), 4),
        max=round(float(q95), 4),
        step=round(float(step), 5),
        description=feat,
        continuous_update=False,
        style={'description_width': '120px'},
        layout=widgets.Layout(width='520px'),
    )

status_label = widgets.Label(value="")

def on_change(change):
    status_label.value = "Computing..."
    vals  = {feat: sliders[feat].value for feat in slider_features}
    preds = predict_grid(**vals)
    with fig.batch_update():
        fig.data[0].z = preds
    status_label.value = ""

for s in sliders.values():
    s.observe(on_change, names='value')

slider_box = widgets.VBox(
    [widgets.HTML("<b style='font-size:13px'>Adjust environmental conditions (Filtered to Water):</b>"),
     status_label]
    + list(sliders.values()),
    layout=widgets.Layout(padding='8px 12px')
)

display(widgets.VBox([fig, slider_box]))


    'data': [{'colorbar': {'thickness': 16, 'title': {'text': 'ChlorA<br>(μg/L)'…